# Encoder-router volume probe — v2 ∪ v3 ∪ v2-100K

**Question.** `architecture_5k_test` closed with a clear finding: post-retrieval operators (llm_a, r1, deep ranking) don't beat l1; only l2 adds real headroom (+0.08, 6× variance collapse). The **router** is the one untouched dimension. Does more label data lift the router's ceiling on the *decisive* stratum, or is that ceiling information-bounded?

**Motivation.** Four label sources: v2 (~5K), v3 (~72K natural), v3-augmented (~100K synthetic-generated, unique query_ids), v2-100K (91K). Runtime union after dedup on `(dataset, query_id)`: **233,247 rows across 46 lanes** — 2.6× the current baseline (91K). v3-augmented's synthetic queries dominate the growth: they carry their own query_ids, so almost none collide with the natural pool. Rather than commit to a joint parquet while the design is unstable, we union them **at runtime** and inject the frame into `TrainingTable` in memory.

**Success criteria.** Compare per-lane *decisive-headroom capture* between the current baseline (91K rows) and the union (233K rows, 2.6× growth) using two arms first: `no_branches` and `shuffled_targets`. Wobble is ±0.03/fit (`project_instrument_validation`), so:
- A consistent >0.03 shift across ≥3 lanes on `no_branches` → volume is real signal; run the `design` arm next.
- No shift → EITHER the router's ceiling is not sample-count-bound OR v3-augmented's synthetic queries don't transfer to natural-query routing. Both are useful findings; the decision cell splits them.

**Non-goals.** Not creating a joint parquet. Not fine-tuning BGE. Not rerunning failed operators. Not the `design` arm until baseline arms clear the gate.

## Colab setup

Run the bootstrap and installation cells below before the experiment imports.
If installation requests a session restart, restart, rerun the bootstrap, and
**skip the installation cell**. This uses the same full-project installation as
the dataset deep dive; pip can report conflicts with Colab's preinstalled packages.
Local runs skip both cells.

The Git checkout does not include data or model weights. These experiments need
more inputs than the deep dive's small `data_colab` bundle. To restore the DVC
artifacts in Colab, run this in a separate cell after installation:

```python
%pip install "dvc[gs]"
!dvc pull -r public src/data.dvc models.dvc
```

Run that command from `/content/search-routing-experiments` (the bootstrap's
working directory). It downloads the full versioned data and model artifacts,
which can occupy many GB. Source access terms still apply.

The cells below retain their existing training settings. Training can take substantial
time; select a Colab runtime with sufficient memory and inspect the retraining flags
before running the full notebook.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Stable across cell reruns and session restarts.
if "google.colab" in sys.modules:
    root = Path("/content/search-routing-experiments")
    if not (root / ".git").exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/qdrant-labs/search-routing-experiments",
            str(root),
        ], check=True)
    os.chdir(root)
    if str(root / "src") not in sys.path:
        sys.path.insert(0, str(root / "src"))


def runtime_value(name: str, default=None):
    """Read a Colab secret when available, otherwise an environment variable."""
    if "google.colab" in sys.modules:
        from google.colab import userdata

        try:
            value = userdata.get(name)
        except userdata.SecretNotFoundError:
            value = None
        if value is not None:
            return value
    return os.environ.get(name, default)


In [ ]:
if "google.colab" in sys.modules:
    root = Path("/content/search-routing-experiments")
    if not root.is_dir():
        raise RuntimeError("Run the Colab bootstrap cell before installing dependencies.")

    # Define submodule path
    submodule_path = root / "src" / "query-taxonomy"

    # Remove the submodule directory if it exists, to ensure a clean state
    if submodule_path.is_dir():
        import shutil
        shutil.rmtree(submodule_path)
        print(f"Removed existing submodule directory: {submodule_path}")
    elif submodule_path.is_file(): # In case it's a gitlink file from a submodule without content
        submodule_path.unlink() # remove the file

    # Instead of git submodule commands, directly clone the repository
    # This bypasses potential issues with git submodule update in shallow clones
    if not submodule_path.is_dir(): # Only clone if it doesn't exist after cleanup
        print(f"Cloning query-taxonomy to {submodule_path}...")
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/qdrant-labs/query-taxonomy.git",
            str(submodule_path)
        ], cwd=root, check=True)
        print("Query-taxonomy cloned successfully.")
    else:
        print(f"Query-taxonomy already present at {submodule_path}.")

    # Explicitly add the directory containing the 'query_taxonomy' package to sys.path.
    # This addresses ModuleNotFoundError issues when 'pip -e' might not correctly
    # register the path in Colab's dynamic environment, especially after chdir/sys.path manipulations.
    if str(submodule_path) not in sys.path:
        sys.path.insert(0, str(submodule_path))
        print(f"Added {submodule_path} to sys.path for direct import.")

    pip_install_cmd = [
        sys.executable, "-m", "pip", "install",
        "-e", str(root / "src" / "query-taxonomy"),
        "-e", str(root),
        "fastembed", # Explicitly install fastembed to resolve ModuleNotFoundError
        "qdrant-client" # Explicitly install qdrant-client
    ]
    print(f"Running pip install: {' '.join(pip_install_cmd)}")
    install_result = subprocess.run(pip_install_cmd, capture_output=True, text=True, check=False)
    print("--- pip install stdout ---")
    print(install_result.stdout)
    print("--- pip install stderr ---")
    print(install_result.stderr)

    if install_result.returncode != 0:
        raise RuntimeError(f"Pip installation failed with exit code {install_result.returncode}")

    print("Installation complete. Restart the Colab session, rerun the bootstrap cell, and continue with the imports.")

In [ ]:
if "google.colab" in sys.modules:
    root = Path("/content/search-routing-experiments")
    data_dir = root / "src" / "data"
    if data_dir.is_dir():
        print(f"{data_dir} already present, skipping dvc pull.")
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "dvc[gs]"], check=True)
        subprocess.run(
            ["dvc", "pull", "-r", "public", "src/data.dvc"],
            cwd=root,
            check=True,
        )


In [1]:
from __future__ import annotations

import os

# torch, sklearn, and lightgbm each ship their own libomp on macOS; loading two
# in one process corrupts OpenMP barriers -> SIGSEGV in the next parallel op
# (seen: kernel death inside torch.ones during EncoderRouter.fit). Must be set
# BEFORE the first import of any of them.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

from pathlib import Path

import numpy as np
import pandas as pd

from encoder_router.evaluate import ARMS, run_arms
from encoder_router.table import KEY, ROUTES
from encoder_router.targets import OUT_DIR
from encoder_router.training import TrainingTable
from hybrid_search_rrf_dataset.labels import DEFAULT_OUT_DIR, AcceptabilityLabels

DATA_DIR = DEFAULT_OUT_DIR.parent
LABEL_SOURCES = {
    "v2": DEFAULT_OUT_DIR / "labels.parquet",
    "v3_augmented": DATA_DIR / "v3" / "augmented" / "labels.parquet",
    "v3": DATA_DIR / "v3" / "labels.parquet",
    "v2_100k": DATA_DIR / "rungs" / "100k-v2" / "labeling" / "labels.parquet",
    # "v2_100k": DATA_DIR / "legb_pilot" / "os_distill_relabel" / "cascade_labels.parquet"
}
# v2-100K wins on (dataset, query_id) collision — it is the current target
# per project_target_v2_100k. Named here so the merge decision is not silent.
WINNER_ORDER = ("v2", "v3", "v3_augmented", "v2_100k")
SEED = 0

for tag, p in LABEL_SOURCES.items():
    print(f"{tag:8s}  {'ok' if p.exists() else 'MISSING':7s}  {p}")

v2        ok       /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/route_labels/labels.parquet
v3_augmented  ok       /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/v3/augmented/labels.parquet
v3        ok       /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/v3/labels.parquet
v2_100k   ok       /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/rungs/100k-v2/labeling/labels.parquet


## Reading it

- **`mean_H` and `lanes_beating_const` for `no_branches` vs `shuffled_targets`** is the
  headline. Within-lane there are no cross-lane contradictions: if the real arm still
  can't clear the shuffled floor and the per-lane constants, the input representation
  is the wall (rarity blindness) and no label work changes that. If it clears them,
  labels + inputs are jointly sufficient and the generic-router problem reduces to
  cross-lane label structure — the consistency census becomes the next artifact.
- **`zipf_input` vs `no_branches`**: whether query-intrinsic rarity features buy
  anything once lane structure is out of the picture.
- **`decisive_only` vs `no_branches`**: the tie-hygiene ablation — whether the 24K
  qrels-ceiling tie rows ("both acceptable" targets) dilute the heads.
- Caveat: within-lane evaluation is a DIAGNOSTIC, not a deployability claim — the
  router sees each lane's own queries at train time, which the generic spec forbids.
  This probe exists to locate the wall, not to ship a number.

## Plan

**Metric — decisive-headroom capture (per arm × per lane):**

$$H = \frac{\text{captured} - \text{const\_dense}}{\text{oracle} - \text{const\_dense}}$$

**Baseline = the per-lane BEST constant** (max of const_dense/sparse/rrf), not const_dense. `project_viability_program` already set this bar: the LR beats const-dense but LOSES to the per-lane best constant, and a const_dense baseline wrongly credits constant-sparse lanes like clerc (served 100% sparse) as router skill. `H = 1` = captured all headroom over that best constant; `H ≤ 0` = no better than it. Lanes where `oracle ≈ best_const` produce NaN and are excluded — the residual/unroutable rows that want a constant, not a classifier.

**Probe arms — every variant has `corpus_branch=False`.** Rationale: `GoldDocProfile.build` skips *lanes* it already has, not query_ids, so union-added query_ids in an existing lane get NaN corpus targets. `_masked_mse` handles that cleanly, but the corpus branch's effective training set doesn't grow — which muddies the volume signal. Turning it off across the panel keeps the axes comparable.

| arm                       | tests |
|---                        |---|
| `no_branches`             | base MLP: does volume lift the raw router? |
| `shuffled_targets`        | permutation floor (guards against distribution shift) |
| `features_input_nocorpus` | taxonomy features as inputs — do they earn keep at scale? |
| `zipf_input_nocorpus`     | serve-safe rarity input — does it help? |
| `lightgbm_nocorpus`       | LightGBM as an alternative learner |

Each arm is trained once and saved in §3b; measurement is §4e `evaluate_holdout` (one held-out lane). No LOO committee, no gate.

## 1. Load each label source and inspect its schema

Before unioning, check that the three sources are actually commensurable. `project_label_artifact_staleness` warned that v3's `pure_rrf` was fetched at depth 1000 vs oracle 50 — a naive concat would fuse rows judged by different rulers.

In [2]:
raw = {tag: pd.read_parquet(path) for tag, path in LABEL_SOURCES.items()}

pd.DataFrame({
    tag: {
        "rows": len(frame),
        "unique_keys": len(frame.drop_duplicates(KEY)),
        "lanes": frame["dataset"].nunique() if "dataset" in frame.columns else np.nan,
        "has_query": "query" in frame.columns,
        "has_score_cols": all(f"score_{r}" in frame.columns for r in ROUTES),
    }
    for tag, frame in raw.items()
}).T

,rows,unique_keys,lanes,has_query,has_score_cols
v2,46142,46142,42,True,True
v3_augmented,107120,107120,40,True,True
v3,71966,71966,28,True,True
v2_100k,91093,91093,46,True,True


In [3]:
# Column overlap — which columns exist in all three vs only in some?
all_cols = sorted(set().union(*(frame.columns for frame in raw.values())))
overlap = pd.DataFrame(
    {tag: [col in frame.columns for col in all_cols] for tag, frame in raw.items()},
    index=all_cols,
)
overlap["in_all"] = overlap.all(axis=1)
overlap[~overlap["in_all"]]  # rows where the sources disagree

,v2,v3_augmented,v3,v2_100k,in_all
cell,True,False,False,False,False
checkable,True,False,False,False,False
provenance,False,True,True,True,False
route_selected,True,False,False,False,False
scored_against,False,True,True,True,False
stage,True,False,False,False,False


## 2. Runtime union — no file, no schema commitment

Ordered concat + `drop_duplicates(keep="last")` — the last source in `WINNER_ORDER` wins on collision. Kept fully in memory; nothing persists.

In [ ]:
from dataset_release import build_91k, build_union_233k

union_raw = build_union_233k(DATA_DIR)
print(f"union rows: {len(union_raw):,} across {union_raw['dataset'].nunique()} lanes (cascade wins collisions)")


In [5]:
def build_table(raw_labels: pd.DataFrame) -> TrainingTable:
    """Bypass TrainingTable's disk read — inject the frame directly into the
    cached_property slot. Same downstream behavior, no parquet on disk."""
    merged = AcceptabilityLabels(raw_labels, tolerance=None).frame()
    missing = merged["query"].isna() | (merged["query"].astype(str) == "")
    frame = merged[~missing].reset_index(drop=True)
    table = TrainingTable()
    table.__dict__["frame"] = frame  # override cached_property before first access
    return table

table_baseline = build_table(build_91k(DATA_DIR))   # current 91K target
table_union = build_table(union_raw)           # ~200K probe pool

print(f"baseline rows: {len(table_baseline.frame):,} across {table_baseline.frame['dataset'].nunique()} lanes")
print(f"union    rows: {len(table_union.frame):,} across {table_union.frame['dataset'].nunique()} lanes")

baseline rows: 91,080 across 46 lanes
union    rows: 233,247 across 46 lanes


## 3. Probe arms — 5 configurations

Five arm configurations, each trained ONCE and saved in §3b: `no_branches` (base MLP),
`shuffled_targets` (permutation floor), `features_input_nocorpus` (taxonomy features as inputs),
`zipf_input_nocorpus` (serve-safe rarity input), `lightgbm_nocorpus` (LightGBM learner). Every arm
has `corpus_branch=False` so they share the same base training set. Measurement is §4e
`evaluate_holdout` (one held-out lane); serving is §4c.

In [6]:
from IPython.display import display
from encoder_router.evaluate import Arm

PROBE_ARMS = (
    Arm("no_branches",             cell_branch=False, corpus_branch=False),
    Arm("shuffled_targets",        shuffle_targets=True, corpus_branch=False),
    Arm("features_input_nocorpus", feature_inputs=True, cell_branch=False, corpus_branch=False),
    Arm("zipf_input_nocorpus",     zipf_inputs=True,    cell_branch=False, corpus_branch=False),
    Arm("zipf_shape_nocorpus",     zipf_inputs=True,    shape_inputs=True, cell_branch=False, corpus_branch=False),
    Arm("lightgbm_nocorpus",       feature_inputs=True, learner="lgbm",
                                   cell_branch=False, corpus_branch=False),
    # LUPI: predict corpus/gold-doc profiles from the query latent (a "hidden corpus
    # representation") and feed it forward to the route layers — serve-safe (no corpus at
    # inference). The one arm that can carry collection-relative signal query-only.
    Arm("design_branches",         cell_branch=True, corpus_branch=True),
    # no-SVD ablation: embedding channel only — does the char-ngram SVD earn its 128 dims,
    # or is it redundant with what BGE already encodes?
    Arm("no_svd_nocorpus",          svd_inputs=False, cell_branch=False, corpus_branch=False),
    # margins-as-targets: regress score_sparse/score_dense (ties kept as small-margin rows,
    # not binarized away); serve = margin-gated argmax of predicted scores (RRF hedge = gate).
    Arm("score_regressor_nocorpus", learner="regressor", cell_branch=False, corpus_branch=False),
    # tie hygiene: both heads are ~82% positive because most rows say "either route is
    # fine". Zero-weighting them leaves only rows that state a preference — the stratum
    # §4e actually scores.
    Arm("decisive_only",            decisive_only=True, cell_branch=False, corpus_branch=False),
)

def _with_derived(df: pd.DataFrame) -> pd.DataFrame:
    """Derive headroom over the per-lane BEST constant (+ served_max) for the eye tests."""
    for col in ("train_val_loss", "train_epochs"):
        if col not in df.columns:
            df = df.assign(**{col: np.nan})
    const_cols = [c for c in ("const_dense", "const_sparse", "const_rrf") if c in df.columns]
    best_const = df[const_cols].max(axis=1)
    served_cols = [c for c in ("served_dense_only", "served_sparse_only", "served_pure_rrf") if c in df.columns]
    served_max = df[served_cols].max(axis=1)
    gap = (df["oracle"] - best_const).replace(0, np.nan)
    return df.assign(
        best_const   = best_const,
        served_max   = served_max,
        headroom     = (df["captured"] - best_const) / gap,
        oracle_ratio = df["captured"] / df["oracle"].replace(0, np.nan),
    )

## 3a. Pick the validation lane FIRST — held out during training (no leakage)

Rank lanes by routable headroom (`oracle - best_const`) and diversity, choose `FAIR_LANE`, and hold
it OUT of §3b training. clerc was rigged (sparse-dominated, ~0.05 gap); a fair lane has a large gap
and a balanced serve mix. Because §3b excludes `FAIR_LANE`, the saved model can be validated on it
honestly in §4e — same model, no leakage.

In [7]:
# 4e-pre — rank lanes by routable headroom (oracle - best_const) and route diversity
MIN_GAP = 0.10             # routable headroom over the best constant a lane must offer
MAX_DOMINANT_SHARE = 0.55  # truth routes must be balanced: no single route may exceed this share
MIN_ROWS = 300             # enough decisive rows for a stable held-out estimate

def rank_lanes(table, min_rows=200):
    dec = table.frame[(table.frame["shape"] == "routes_differ") & table.frame["serve"].notna()]
    rows = []
    for lane, g in dec.groupby("dataset"):
        if len(g) < min_rows:
            continue
        sc = {r: g[f"score_{r}"].to_numpy() for r in ROUTES}
        consts = {r: float(sc[r].mean()) for r in ROUTES}
        best_const = max(consts.values())
        oracle = float(np.column_stack([sc[r] for r in ROUTES]).max(1).mean())
        mix = g["serve"].value_counts(normalize=True)
        rows.append({"lane": lane, "n": len(g), "best_const": best_const, "oracle": oracle,
                     "routable_gap": oracle - best_const,
                     "best_route": max(consts, key=consts.get),
                     "dominant_serve": mix.idxmax(), "dominant_share": float(mix.max())})
    return pd.DataFrame(rows).sort_values("routable_gap", ascending=False).reset_index(drop=True)

ranked = rank_lanes(table_union)
print("lanes by ROUTABLE headroom (big gap + low dominant_share = fair router test):")
display(ranked.round(3).head(15))
print("clerc (the bad default) for contrast:")
display(ranked[ranked["lane"] == "clerc"].round(3))

_fair = ranked[(ranked["routable_gap"] > MIN_GAP)
               & (ranked["dominant_share"] < MAX_DOMINANT_SHARE)
               & (ranked["n"] >= MIN_ROWS)]
FAIR_LANE = _fair.iloc[0]["lane"] if len(_fair) else ranked.iloc[0]["lane"]
print(f"\nfair candidates (gap>{MIN_GAP}, dominant_share<{MAX_DOMINANT_SHARE}, n>={MIN_ROWS}): "
      f"{_fair['lane'].tolist()[:8]}")
print(f"-> FAIR_LANE = {FAIR_LANE!r}  (used by §3b/§4e; override freely, or loop over _fair['lane'])")

lanes by ROUTABLE headroom (big gap + low dominant_share = fair router test):


,lane,n,best_const,oracle,routable_gap,best_route,dominant_serve,dominant_share
0,trec-dl-2022,524,0.432,0.698,0.266,sparse_only,sparse_only,0.712
1,freshstack-laravel,4548,0.467,0.651,0.184,dense_only,sparse_only,0.594
2,crumb-clinical-trial,4354,0.430,0.608,0.178,pure_rrf,sparse_only,0.737
3,bright-earth-science,704,0.501,0.676,0.175,pure_rrf,sparse_only,0.615
4,bright-biology,1764,0.441,0.614,0.173,pure_rrf,sparse_only,0.656
5,clerc,8167,0.499,0.664,0.166,sparse_only,sparse_only,0.819
6,rarb-math,4337,0.479,0.644,0.165,pure_rrf,sparse_only,0.719
7,finder,2013,0.426,0.583,0.157,pure_rrf,sparse_only,0.785
8,freshstack-yolo,1899,0.455,0.610,0.155,dense_only,sparse_only,0.594
9,scirgen-geo-en,20924,0.366,0.515,0.149,pure_rrf,sparse_only,0.827


clerc (the bad default) for contrast:


,lane,n,best_const,oracle,routable_gap,best_route,dominant_serve,dominant_share
5,clerc,8167,0.499,0.664,0.166,sparse_only,sparse_only,0.819



fair candidates (gap>0.1, dominant_share<0.55, n>=300): []
-> FAIR_LANE = 'trec-dl-2022'  (used by §3b/§4e; override freely, or loop over _fair['lane'])


## 3b. Train & save ONE model per arm — holding out FAIR_LANE

Per arm, fit ONE model on all lanes EXCEPT `FAIR_LANE` (arm-faithful inputs / branches / learner)
and save to `BASE/<arm.name>/`. The held-out lane is recorded in `meta.json`, so §4e validates the
exact saved model on it with no leakage. `FORCE_RETRAIN=True` overwrites; re-runs skip saved arms.

In [8]:
# 3b — per-arm: train ONE model per arm on all lanes EXCEPT FAIR_LANE (held out), save each
import json as _json, joblib, time as _time
from pathlib import Path as _P
from encoder_router.evaluate import LaneCV, tuned_thresholds
from encoder_router.model import EncoderRouter
from encoder_router.table import (EMBEDDING_PREFIXES, HEAD_ROUTES, LexicalShape,
                                   NgramSvd, ZipfStats, serve_from_probabilities)
from encoder_router.training import QueryEmbeddings

TRAIN_VERSION = "2026-09-10-lanesplit"   # bump on ANY training-logic change -> invalidates cache
# Hand thresholds that deliberately override the tuner (domain prior > near-noise tuner edge):
# zipf_input's dense head capped so DHA-class semantic queries route dense. Applied in train_arm
# before save, so it survives retrains and every loader (classify, §4e) reads it from disk.
THRESHOLD_OVERRIDES = {"zipf_input_nocorpus": [0.3, 0.75], "zipf_shape_nocorpus": [0.3, 0.9]}

def _zstats(a):
    m = a.mean(0); s = a.std(0); s[s == 0] = 1.0
    return m.astype(np.float32), s.astype(np.float32)

def train_arm(table, arm, base_dir, holdout_lane=None, seed=SEED):
    """Train ONE model faithful to `arm` on all lanes EXCEPT holdout_lane; save to base_dir/arm.name."""
    t0 = _time.perf_counter()
    def log(msg): print(f"  [{arm.name}] +{_time.perf_counter()-t0:6.1f}s  {msg}", flush=True)

    frame = table.frame
    pool_mask = np.ones(len(frame), bool) if holdout_lane is None else (frame["dataset"] != holdout_lane).to_numpy()
    pool = np.flatnonzero(pool_mask)
    log(f"START learner={arm.learner} feature_inputs={arm.feature_inputs} zipf_inputs={arm.zipf_inputs}")
    log(f"holdout_lane={holdout_lane!r} | train on {len(pool):,}/{len(frame):,} rows "
        f"({frame.loc[pool_mask, 'dataset'].nunique()} lanes)")
    emb = QueryEmbeddings(arm.embedding_model).matrix(frame); log(f"embeddings {emb.shape}")
    blocks, stats, svd = [emb], {}, None
    if arm.svd_inputs:
        svd = NgramSvd(seed=seed).fit(frame.loc[pool_mask, "query"]); log("ngram-SVD fit (train lanes only)")
        blocks.append(svd.transform(frame["query"]))
    else:
        log("no-SVD arm: embedding channel only")
    if arm.feature_inputs:
        log("assembling taxonomy feature inputs...")
        f = table.feature_matrix.to_numpy(np.float32); m, s = _zstats(f[pool_mask])
        stats["feat"] = (m, s); blocks.append((f - m) / s); log(f"feature inputs {f.shape}")
    if arm.zipf_inputs:
        z = ZipfStats().frame(frame["query"]).to_numpy(np.float32); m, s = _zstats(z[pool_mask])
        stats["zipf"] = (m, s); blocks.append((z - m) / s); log(f"zipf inputs {z.shape}")
    if arm.shape_inputs:
        sh = LexicalShape().frame(frame["query"]).to_numpy(np.float32); m, s = _zstats(sh[pool_mask])
        stats["shape"] = (m, s); blocks.append((sh - m) / s); log(f"shape inputs {sh.shape}")
    x = np.concatenate(blocks, axis=1).astype(np.float32)
    route = table.route_targets().to_numpy(np.float32)
    log(f"input matrix x={x.shape}")
    cv = LaneCV(table, seed=seed)
    cell_t, corpus_t, feat_t = cv._targets(arm, pool_mask)
    weights = cv.row_weights(arm)
    log(f"row weights: {int((weights[pool_mask] > 0).sum()):,}/{len(pool):,} nonzero in pool, "
        f"mean {weights[pool_mask].mean():.3f}")
    log(f"branches: cell={None if cell_t is None else cell_t.shape} "
        f"corpus={None if corpus_t is None else corpus_t.shape} feature={None if feat_t is None else feat_t.shape}")
    if arm.shuffle_targets:                     # real permutation FLOOR: shuffle ROUTE labels in-pool
        rp = np.random.default_rng(seed + 1).permutation(len(pool))
        route = route.copy(); route[pool] = route[pool][rp]
        log("shuffled ROUTE labels within train pool (permutation floor; thresholds still use real labels)")

    p = _P(base_dir) / arm.name; p.mkdir(parents=True, exist_ok=True)
    if arm.learner == "lgbm":
        from lightgbm import LGBMClassifier
        models = []
        for i, head in enumerate(HEAD_ROUTES):
            ok = (~np.isnan(route[:, i])) & pool_mask; pos = route[ok, i].sum()
            log(f"lgbm head '{head}': fit on {int(ok.sum()):,} rows ({int(pos):,} positive)")
            models.append(LGBMClassifier(n_estimators=400, learning_rate=0.05, n_jobs=1,
                random_state=seed, verbose=-1,
                scale_pos_weight=float(np.clip((ok.sum() - pos) / max(pos, 1), 1, 100))
                ).fit(x[ok], route[ok, i], sample_weight=weights[ok]))
        joblib.dump(models, p / "lgbm.joblib"); log("lgbm models saved")
        probs_tr = pd.DataFrame(np.column_stack([m.predict_proba(x[pool])[:, 1] for m in models]), columns=HEAD_ROUTES)
    elif arm.learner == "regressor":
        from lightgbm import LGBMRegressor
        ans = pool_mask & frame["serve"].notna().to_numpy()   # answerable rows: ties kept as small-margin targets
        regs = []
        for head in HEAD_ROUTES:
            y = frame[f"score_{head}"].to_numpy(np.float32)
            log(f"regressor head '{head}': fit on {int(ans.sum()):,} rows (margins as targets)")
            regs.append(LGBMRegressor(n_estimators=400, learning_rate=0.05, n_jobs=1,
                random_state=seed, verbose=-1).fit(x[ans], y[ans], sample_weight=weights[ans]))
        joblib.dump(regs, p / "regressor.joblib"); log("regressor models saved")
        probs_tr = pd.DataFrame(np.column_stack([r.predict(x[pool]) for r in regs]), columns=HEAD_ROUTES)
    else:
        fi, vi = cv._fit_val_split(pool_mask)   # whole LANES held out for early stopping
        log(f"mlp fit: {len(fi):,} rows / {frame.iloc[fi]['dataset'].nunique()} lanes  ->  "
            f"val {len(vi):,} rows / {frame.iloc[vi]['dataset'].nunique()} lanes "
            f"{sorted(frame.iloc[vi]['dataset'].unique())}")
        er = EncoderRouter(seed=seed).fit(x[fi], route[fi],
            None if cell_t is None else cell_t[fi], None if corpus_t is None else corpus_t[fi],
            None if feat_t is None else feat_t[fi],
            route_weights=weights[fi],
            x_val=x[vi], val_route_targets=route[vi], val_route_weights=weights[vi])
        log(f"mlp done: {len(er.history)} epochs run, best_epoch={er.best_epoch}, "
            f"best_val={getattr(er, 'best_val_loss', float('nan')):.4f}")
        er.save(p / "router.pt"); log("router.pt saved"); probs_tr = er.probabilities(x[pool])
    if arm.learner == "regressor":
        thr = np.zeros(len(HEAD_ROUTES), np.float32)   # argmax of predicted scores; RRF_DELTA is the margin gate
        log("regressor: thresholds=0 (serve = margin-gated argmax; hedge = RRF_DELTA)")
    else:
        thr = tuned_thresholds(probs_tr, frame.loc[pool_mask]); log(f"tuned thresholds {np.round(thr, 3).tolist()}")
    if arm.name in THRESHOLD_OVERRIDES:
        thr = np.asarray(THRESHOLD_OVERRIDES[arm.name], np.float32)
        log(f"threshold OVERRIDE -> {thr.tolist()} (hand cap over tuner)")
    if svd is not None:
        svd.save(p / "svd.joblib")
    np.save(p / "thresholds.npy", thr)
    for k, (m, s) in stats.items():
        np.save(p / f"{k}_mean.npy", m); np.save(p / f"{k}_std.npy", s)
    mix = pd.Series(serve_from_probabilities(probs_tr, thr)).value_counts(normalize=True).round(2).to_dict()
    log(f"served mix (train lanes): {mix}")
    (p / "meta.json").write_text(_json.dumps(
        {"arm": arm.name, "learner": arm.learner, "embedding_model": arm.embedding_model,
         "prefix": EMBEDDING_PREFIXES.get(arm.embedding_model, ""),
         "feature_inputs": arm.feature_inputs, "zipf_inputs": arm.zipf_inputs,
         "shape_inputs": arm.shape_inputs, "svd_inputs": arm.svd_inputs,
         "decisive_only": arm.decisive_only,
         "best_epoch": int(er.best_epoch) if arm.learner not in ("lgbm", "regressor") else None,
         "serve_safe": not arm.feature_inputs, "holdout_lane": holdout_lane,
         "trained_at": _time.strftime("%Y-%m-%d %H:%M:%S"), "train_version": TRAIN_VERSION}))
    log(f"SAVED -> {p}  (held out {holdout_lane!r}, total {_time.perf_counter()-t0:.1f}s)")
    return p

BASE = OUT_DIR / "classifiers_union_200k"     # per-arm models under BASE/<arm.name>/
FORCE_RETRAIN = True                          # True -> retrain & OVERWRITE every arm, even if saved

def _cache_ok(arm, base, holdout):
    """A saved arm is reusable ONLY if its manifest matches this run's holdout lane + config.
    A bare exists() check would silently reuse pre-holdout (all-lanes, leaked) models."""
    p = base / arm.name; mp = p / "meta.json"
    if not mp.exists():
        return False
    try:
        m = _json.loads(mp.read_text())
    except Exception:
        return False
    weights = {"lgbm": "lgbm.joblib", "regressor": "regressor.joblib"}.get(arm.learner, "router.pt")
    return (m.get("train_version") == TRAIN_VERSION and m.get("holdout_lane") == holdout
            and m.get("arm") == arm.name and m.get("learner") == arm.learner
            and m.get("feature_inputs") == arm.feature_inputs
            and m.get("zipf_inputs") == arm.zipf_inputs
            and m.get("shape_inputs", False) == arm.shape_inputs
            and m.get("svd_inputs", True) == arm.svd_inputs
            and m.get("decisive_only", False) == arm.decisive_only
            and (p / weights).exists())

_todo   = list(PROBE_ARMS) if FORCE_RETRAIN else [a for a in PROBE_ARMS if not _cache_ok(a, BASE, FAIR_LANE)]
_cached = [] if FORCE_RETRAIN else [a.name for a in PROBE_ARMS if _cache_ok(a, BASE, FAIR_LANE)]
print(f"per-arm training | holdout={FAIR_LANE!r} | table_union {len(table_union.frame):,} rows | FORCE_RETRAIN={FORCE_RETRAIN}")
print(f"  cached (skip): {_cached or '-'}")
print(f"  to train:      {[a.name for a in _todo] or '-'}", flush=True)
for _n, _a in enumerate(_todo, 1):
    print(f"===== arm {_n}/{len(_todo)}: {_a.name} =====", flush=True)
    train_arm(table_union, _a, BASE, holdout_lane=FAIR_LANE)
    print(flush=True)
print("all arms saved." if _todo else "nothing to train - all arms cached.")

per-arm training | holdout='trec-dl-2022' | table_union 233,247 rows | FORCE_RETRAIN=True
  cached (skip): -
  to train:      ['no_branches', 'shuffled_targets', 'features_input_nocorpus', 'zipf_input_nocorpus', 'zipf_shape_nocorpus', 'lightgbm_nocorpus', 'design_branches', 'no_svd_nocorpus', 'score_regressor_nocorpus', 'decisive_only']
===== arm 1/10: no_branches =====
  [no_branches] +   0.0s  START learner=mlp feature_inputs=False zipf_inputs=False
  [no_branches] +   0.0s  holdout_lane='trec-dl-2022' | train on 232,062/233,247 rows (45 lanes)
  [no_branches] +   5.9s  embeddings (233247, 384)
  [no_branches] +  43.5s  ngram-SVD fit (train lanes only)
  [no_branches] +  55.6s  input matrix x=(233247, 512)
  [no_branches] +  55.6s  row weights: 232,062/232,062 nonzero in pool, mean 0.776
  [no_branches] +  55.6s  branches: cell=None corpus=None feature=None
  [no_branches] +  55.7s  mlp fit: 208,856 rows / 45 lanes  ->  val 23,206 rows / 45 lanes ['antique', 'beir-nfcorpus', 'beir-to

fit:   0%|          | 0/200 [00:00<?, ?it/s]

  [no_branches] +  80.3s  mlp done: 24 epochs run, best_epoch=3, best_val=0.2413
  [no_branches] +  80.4s  router.pt saved
  [no_branches] +  81.2s  tuned thresholds [0.25, 0.95]
  [no_branches] +  81.2s  served mix (train lanes): {'sparse_only': 0.86, 'dense_only': 0.14}
  [no_branches] +  81.2s  SAVED -> /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/encoder_router/classifiers_union_200k/no_branches  (held out 'trec-dl-2022', total 81.2s)

===== arm 2/10: shuffled_targets =====
  [shuffled_targets] +   0.0s  START learner=mlp feature_inputs=False zipf_inputs=False
  [shuffled_targets] +   0.0s  holdout_lane='trec-dl-2022' | train on 232,062/233,247 rows (45 lanes)
  [shuffled_targets] +   5.5s  embeddings (233247, 384)
  [shuffled_targets] +  45.7s  ngram-SVD fit (train lanes only)
  [shuffled_targets] +  58.0s  input matrix x=(233247, 512)
extracting features for 107765 unindexed queries
  [shuffled_targets] + 199.2s  row weights: 232,062/232,062 nonzero in pool, mean 0.7

fit:   0%|          | 0/200 [00:00<?, ?it/s]

  [shuffled_targets] + 238.5s  mlp done: 21 epochs run, best_epoch=0, best_val=0.2391
  [shuffled_targets] + 238.5s  router.pt saved
  [shuffled_targets] + 239.3s  tuned thresholds [0.05, 0.55]
  [shuffled_targets] + 239.4s  served mix (train lanes): {'sparse_only': 1.0}
  [shuffled_targets] + 239.4s  SAVED -> /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/encoder_router/classifiers_union_200k/shuffled_targets  (held out 'trec-dl-2022', total 239.4s)

===== arm 3/10: features_input_nocorpus =====
  [features_input_nocorpus] +   0.0s  START learner=mlp feature_inputs=True zipf_inputs=False
  [features_input_nocorpus] +   0.0s  holdout_lane='trec-dl-2022' | train on 232,062/233,247 rows (45 lanes)
  [features_input_nocorpus] +   6.1s  embeddings (233247, 384)
  [features_input_nocorpus] +  47.5s  ngram-SVD fit (train lanes only)
  [features_input_nocorpus] +  60.7s  assembling taxonomy feature inputs...
  [features_input_nocorpus] +  60.9s  feature inputs (233247, 83)
  [featu

...

  [design_branches] + 125.6s  mlp done: 35 epochs run, best_epoch=14, best_val=0.2399
  [design_branches] + 125.6s  router.pt saved
  [design_branches] + 126.6s  tuned thresholds [0.25, 0.95]
  [design_branches] + 126.6s  served mix (train lanes): {'sparse_only': 0.84, 'dense_only': 0.16}
  [design_branches] + 126.6s  SAVED -> /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/encoder_router/classifiers_union_200k/design_branches  (held out 'trec-dl-2022', total 126.6s)

===== arm 8/10: no_svd_nocorpus =====
  [no_svd_nocorpus] +   0.0s  START learner=mlp feature_inputs=False zipf_inputs=False
  [no_svd_nocorpus] +   0.0s  holdout_lane='trec-dl-2022' | train on 232,062/233,247 rows (45 lanes)
  [no_svd_nocorpus] +   7.3s  embeddings (233247, 384)
  [no_svd_nocorpus] +   7.3s  no-SVD arm: embedding channel only
  [no_svd_nocorpus] +   7.6s  input matrix x=(233247, 384)
  [no_svd_nocorpus] +   7.6s  row weights: 232,062/232,062 nonzero in pool, mean 0.776
  [no_svd_nocorpus] +   7

fit:   0%|          | 0/200 [00:00<?, ?it/s]

  [no_svd_nocorpus] +  32.2s  mlp done: 25 epochs run, best_epoch=4, best_val=0.2427
  [no_svd_nocorpus] +  32.2s  router.pt saved
  [no_svd_nocorpus] +  33.0s  tuned thresholds [0.25, 0.95]
  [no_svd_nocorpus] +  33.0s  served mix (train lanes): {'sparse_only': 0.88, 'dense_only': 0.12}
  [no_svd_nocorpus] +  33.0s  SAVED -> /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/encoder_router/classifiers_union_200k/no_svd_nocorpus  (held out 'trec-dl-2022', total 33.0s)

===== arm 9/10: score_regressor_nocorpus =====
  [score_regressor_nocorpus] +   0.0s  START learner=regressor feature_inputs=False zipf_inputs=False
  [score_regressor_nocorpus] +   0.0s  holdout_lane='trec-dl-2022' | train on 232,062/233,247 rows (45 lanes)
  [score_regressor_nocorpus] +   5.8s  embeddings (233247, 384)
  [score_regressor_nocorpus] +  38.5s  ngram-SVD fit (train lanes only)
  [score_regressor_nocorpus] +  50.1s  input matrix x=(233247, 512)
  [score_regressor_nocorpus] +  50.1s  row weights: 232,

fit:   0%|          | 0/200 [00:00<?, ?it/s]

  [decisive_only] +  74.9s  mlp done: 25 epochs run, best_epoch=4, best_val=0.2450
  [decisive_only] +  74.9s  router.pt saved
  [decisive_only] +  75.5s  tuned thresholds [0.2, 0.95]
  [decisive_only] +  75.5s  served mix (train lanes): {'sparse_only': 0.88, 'dense_only': 0.12}
  [decisive_only] +  75.5s  SAVED -> /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/encoder_router/classifiers_union_200k/decisive_only  (held out 'trec-dl-2022', total 75.5s)

all arms saved.


## 3c. Capacity sweep — optimization wall or information wall?

The base arm stops improving ~epoch 4 of a 200-epoch budget. That is ~1,000 AdamW steps on a
172K-param net over 209K rows, which is convergence, not an under-trained schedule. This cell
settles which wall it is: fit the same inputs at 32x spread in capacity, plus one 10x-slower
LR run.

- **`best_val` flat across the sweep** -> the *inputs* are the ceiling. No amount of schedule
  or regularization tuning moves it, and the epoch count is a red herring (`project_router_input_ceiling`).
- **`best_val` falls with capacity** -> a real optimization/capacity problem worth tuning.

The `lr=1e-4` row is the cosmetic-vs-real check: it should run many more epochs and land at
roughly the same `best_val`. A longer curve with the same floor is not progress.

In [9]:
# 3c - capacity sweep on the base arm (embedding + ngram-SVD). Same lane-held val split as 3b,
# so best_val here is comparable to the numbers 3b logs.
SWEEP = (
    dict(hidden=32,   latent=16,  dropout=0.2, weight_decay=1e-5, lr=1e-3),
    dict(hidden=64,   latent=32,  dropout=0.2, weight_decay=1e-5, lr=1e-3),
    dict(hidden=256,  latent=128, dropout=0.2, weight_decay=1e-5, lr=1e-3),   # the shipped config
    dict(hidden=1024, latent=256, dropout=0.4, weight_decay=1e-2, lr=1e-3),   # 3x params + heavy reg
    dict(hidden=256,  latent=128, dropout=0.2, weight_decay=1e-5, lr=1e-4),   # 10x slower: longer curve, same floor?
)

_arm    = PROBE_ARMS[0]                                    # no_branches: no privileged branches
_frame  = table_union.frame
_pool   = (_frame["dataset"] != FAIR_LANE).to_numpy()
_emb    = QueryEmbeddings(_arm.embedding_model).matrix(_frame)
_svd    = NgramSvd(seed=SEED).fit(_frame.loc[_pool, "query"])
_x      = np.concatenate([_emb, _svd.transform(_frame["query"])], axis=1).astype(np.float32)
_route  = table_union.route_targets().to_numpy(np.float32)
_cv     = LaneCV(table_union, seed=SEED)
_w      = _cv.row_weights(_arm)
_fi, _vi = _cv._fit_val_split(_pool)
print(f"sweep on x={_x.shape} | fit {len(_fi):,} rows / val {len(_vi):,} rows over "
      f"{_frame.iloc[_vi]['dataset'].nunique()} held lanes {sorted(_frame.iloc[_vi]['dataset'].unique())}",
      flush=True)

_rows = []
for _cfg in SWEEP:
    _er = EncoderRouter(seed=SEED, **_cfg).fit(
        _x[_fi], _route[_fi], None, None, None, route_weights=_w[_fi],
        x_val=_x[_vi], val_route_targets=_route[_vi], val_route_weights=_w[_vi])
    _n = sum(t.numel() for t in _er.net.parameters())
    _rows.append({**_cfg, "params": _n, "epochs_run": len(_er.history),
                  "best_epoch": _er.best_epoch, "best_val": _er.best_val_loss})
    print(f"  hidden={_cfg['hidden']:>5} lr={_cfg['lr']:<7g} params={_n:>9,}  "
          f"epochs={len(_er.history):>3}  best_epoch={_er.best_epoch:>3}  "
          f"best_val={_er.best_val_loss:.4f}", flush=True)

_sw = pd.DataFrame(_rows)
_spread = _sw["best_val"].max() - _sw["best_val"].min()
print(f"\nbest_val spread across 32x capacity: {_spread:.4f}")
print("  < ~0.01 -> INFORMATION wall: the inputs cap it, the epoch count is a red herring")
print("  > ~0.03 -> OPTIMIZATION wall: capacity/schedule tuning is worth spending on")
display(_sw.round(4))

sweep on x=(233247, 512) | fit 208,856 rows / val 23,206 rows over 45 held lanes ['antique', 'beir-nfcorpus', 'beir-touche-2020', 'bright-aops', 'bright-biology', 'bright-earth-science', 'bright-economics', 'bright-leetcode', 'bright-pony', 'bright-psychology', 'bright-robotics', 'bright-stackoverflow', 'bright-sustainable-living', 'bright-theoremqa-questions', 'bright-theoremqa-theorems', 'clerc', 'crumb-clinical-trial', 'crumb-code-retrieval', 'crumb-legal-qa', 'crumb-paper-retrieval', 'crumb-set-operation-entity-retrieval', 'crumb-stack-exchange', 'crumb-theorem-retrieval', 'crumb-tip-of-the-tongue', 'dbpedia-entity', 'finder', 'freshstack-angular', 'freshstack-godot', 'freshstack-langchain', 'freshstack-laravel', 'freshstack-yolo', 'gooaq', 'limit', 'lotte-technology-forum', 'lotte-technology-search', 'miracl-en-dev', 'msmarco-passage-dev', 'orcas', 'quest', 'rarb-code', 'rarb-math', 'scirgen-geo-en', 'techqa', 'wands', 'webfaq-eng']


fit:   0%|          | 0/200 [00:00<?, ?it/s]

  hidden=   32 lr=0.001   params=   18,162  epochs= 26  best_epoch=  5  best_val=0.2425


fit:   0%|          | 0/200 [00:00<?, ?it/s]

  hidden=   64 lr=0.001   params=   37,154  epochs= 25  best_epoch=  4  best_val=0.2415


...

fit:   0%|          | 0/200 [00:00<?, ?it/s]

  hidden= 1024 lr=0.001   params=  804,290  epochs= 25  best_epoch=  4  best_val=0.2406


fit:   0%|          | 0/200 [00:00<?, ?it/s]

  hidden=  256 lr=0.0001  params=  172,610  epochs= 40  best_epoch= 19  best_val=0.2417

best_val spread across 32x capacity: 0.0019
  < ~0.01 -> INFORMATION wall: the inputs cap it, the epoch count is a red herring
  > ~0.03 -> OPTIMIZATION wall: capacity/schedule tuning is worth spending on


,hidden,latent,dropout,weight_decay,lr,params,epochs_run,best_epoch,best_val
0,32,16,0.2,0.00,0.0010,18162,26,5,0.2425
1,64,32,0.2,0.00,0.0010,37154,25,4,0.2415
2,256,128,0.2,0.00,0.0010,172610,24,3,0.2413
3,1024,256,0.4,0.01,0.0010,804290,25,4,0.2406
4,256,128,0.2,0.00,0.0001,172610,40,19,0.2417


## 4b. Eye test — per-lane, then read the actual queries

The aggregate says volume didn't help (headroom over the best per-lane constant, degenerate lanes flagged by served_max). Before trusting that, look at it two ways:

- **A (per-lane, free):** which lanes moved, and did the union push lanes to serve dense more
  (the collapse-to-constant / synthetic-poison signature)? Reads the persisted parquets.
- **B (per-query, one refit):** for a lane picked from A, read the real queries where the router
  disagrees with the truth-best route on DECISIVE rows. Swap `table_baseline`→`table_union` to
  see whether volume changed the actual calls on the same lane.

Read both through the LOO lens: the held-lane router was trained on the OTHER 45 lanes, so a
disagreement often means "no training lane taught this lane's shape" ([[project_shape_lane_not_cell]]:
outcome shape is lane-bound), not that the model is weak.

In [10]:
# EYE TEST A — per-lane: volume delta + collapse-to-ANY-constant (from persisted parquets)
def _lane_rows(tag):
    df = _with_derived(pd.read_parquet(OUT_DIR / f"arm_results_volume_{tag}.parquet"))
    return df[df["arm"] == "no_branches"].set_index("lane")

_b, _u = _lane_rows("baseline_91k"), _lane_rows("union_200k")
_lanes = _b.index.intersection(_u.index)
eye = pd.DataFrame({
    "gap":         (_b["oracle"] - _b["best_const"]).loc[_lanes],   # routable headroom over BEST constant
    "H_base":      _b["headroom"].loc[_lanes],
    "H_union":     _u["headroom"].loc[_lanes],
    "smax_base":   _b["served_max"].loc[_lanes],                    # dominant-route share (>=0.95 => constant)
    "smax_union":  _u["served_max"].loc[_lanes],
})
eye["dH"] = eye["H_union"] - eye["H_base"]
eye = eye[eye["gap"] > 0.02]   # material floor: below this there is ~no routable headroom (H explodes)
print(f"lanes with real headroom (gap>0.02): {len(eye)} of {len(_lanes)} | "
      f"improved >0.03: {int((eye['dH']>0.03).sum())} | worse <-0.03: {int((eye['dH']<-0.03).sum())}")
print(f"degenerate (served_max>=0.95, a constant in disguise): "
      f"base {int((eye['smax_base']>=0.95).sum())} | union {int((eye['smax_union']>=0.95).sum())} lanes")
print("\nworst regressions (pick a lane for EYE TEST B):")
display(eye.sort_values("dH").head(8).round(3))
print("best gains:")
display(eye.sort_values("dH").tail(8).round(3))

lanes with real headroom (gap>0.02): 44 of 46 | improved >0.03: 22 | worse <-0.03: 18
degenerate (served_max>=0.95, a constant in disguise): base 13 | union 3 lanes

worst regressions (pick a lane for EYE TEST B):


,gap,H_base,H_union,smax_base,smax_union,dH
lane,,,,,,
crumb-theorem-retrieval,0.049,0.269,-0.563,0.842,0.737,-0.833
beir-touche-2020,0.022,-2.676,-3.421,0.592,0.735,-0.746
freshstack-yolo,0.100,-0.224,-0.779,0.972,0.613,-0.555
bright-stackoverflow,0.092,-0.319,-0.792,0.860,0.616,-0.474
bright-earth-science,0.147,-0.431,-0.804,0.931,0.724,-0.373
freshstack-angular,0.161,0.003,-0.285,0.991,0.544,-0.288
freshstack-langchain,0.125,-0.416,-0.655,1.000,0.515,-0.240
bright-aops,0.029,0.046,-0.175,0.875,0.891,-0.221


best gains:


,gap,H_base,H_union,smax_base,smax_union,dH
lane,,,,,,
webfaq-eng,0.047,-1.875,-1.333,0.679,0.563,0.542
beir-nfcorpus,0.065,-1.682,-1.028,0.564,0.646,0.655
crumb-paper-retrieval,0.033,-3.232,-2.561,1.000,0.947,0.671
dbpedia-entity,0.036,-1.334,-0.607,0.652,0.593,0.727
crumb-stack-exchange,0.079,-1.701,-0.952,0.900,0.583,0.748
crumb-legal-qa,0.061,-3.830,-2.890,0.880,0.835,0.940
trec-dl-2022,0.048,-1.631,-0.060,0.707,0.595,1.571
bright-pony,0.047,-3.546,-1.582,0.791,0.681,1.965


In [11]:
# EYE TEST B — read the real queries the router gets wrong on a chosen lane (one refit)
from encoder_router.evaluate import LaneCV

def query_drilldown(table, lane, arm=PROBE_ARMS[0], seed=SEED):
    """Refit one LOO fold and return per-query rows with truth vs router serve."""
    cv = LaneCV(table, seed=seed)
    frame = table.frame
    emb = QueryEmbeddings(arm.embedding_model).matrix(frame)
    train = (frame["dataset"] != lane).to_numpy()
    x = cv._inputs(arm, frame, emb, train)
    route = table.route_targets().to_numpy(dtype=np.float32)
    served, _thr, _fit = cv._served(arm, x, route, train, frame)
    test = frame[~train][["dataset", "query_id", "query", "shape", "serve",
                          "score_dense_only", "score_sparse_only", "score_pure_rrf"]].copy()
    test["served"] = served
    return test

LANE = "clerc"                       # <- set from EYE TEST A (a regressed lane WITH headroom)
COND = table_baseline                # <- swap to table_union to see if volume changed the calls
dd = query_drilldown(COND, LANE)

differ = dd[(dd["shape"] == "routes_differ") & dd["serve"].notna()]
wrong = differ[differ["served"] != differ["serve"]]
print(f"lane={LANE} ({'baseline' if COND is table_baseline else 'union'}): "
      f"{len(dd)} test rows | {len(differ)} decisive | router wrong on {len(wrong)} "
      f"({len(wrong)/max(len(differ),1):.0%})")
print(f"served mix: {dd['served'].value_counts(normalize=True).round(2).to_dict()}  "
      f"vs truth: {differ['serve'].value_counts(normalize=True).round(2).to_dict()}")
pd.set_option("display.max_colwidth", 90)
print("\ndecisive rows where router != truth (read the queries):")
display(wrong[["query", "serve", "served",
               "score_dense_only", "score_sparse_only", "score_pure_rrf"]].head(20))

fit:   0%|          | 0/200 [00:00<?, ?it/s]

lane=clerc (baseline): 2049 test rows | 1472 decisive | router wrong on 129 (9%)
served mix: {'sparse_only': 1.0, 'dense_only': 0.0}  vs truth: {'sparse_only': 0.91, 'dense_only': 0.07, 'pure_rrf': 0.02}

decisive rows where router != truth (read the queries):


,query,serve,served,score_dense_only,score_sparse_only,score_pure_rrf
1901,"140, 65 S.Ct. 161, 89 L.Ed. 124 (1944)). “That power to persuade depends on the thorou...",dense_only,sparse_only,1.000000,0.106862,1.000000
1967,passage the wind was more or less on the bow. To accommodate wind and sea the course w...,dense_only,sparse_only,1.000000,0.129203,0.189279
1976,"innocence. Here, there is no conflict with such “amenities,” as appellants do not cont...",dense_only,sparse_only,1.000000,0.189279,1.000000
1984,and then went on to resolve ENBC’s section 503(b)(1)(A) claim. The BAP determined that...,pure_rrf,sparse_only,0.189279,0.189279,1.000000
1992,there is a significant change in the useful life of an asset and there is a clear and ...,dense_only,sparse_only,1.000000,0.129203,1.000000
2007,"""292, 153 P.2d 647 (1944), the Utah Supreme Court rejected the defendants’ free exerci...",dense_only,sparse_only,1.000000,0.150000,1.000000
2012,defendant personally at certain critical times such as in deciding whether to accept a...,dense_only,sparse_only,1.000000,0.129203,1.000000
2013,that he will not be penalized for exercising those rights by remaining silent. Id. at ...,dense_only,sparse_only,1.000000,0.150000,0.189279
2014,The language also has the effect of removing the so-called “duty to sit” which has bec...,dense_only,sparse_only,1.000000,0.150000,1.000000
2041,"Alvarez, 679 F.3d 583, 589 (7th Cir. 2012). This appeal raises purely legal questions....",pure_rrf,sparse_only,0.150000,0.189279,1.000000


## 4c. Load a saved arm & classify (no training here — models come from §3b)

`load_classifier(BASE/"<arm>")` reloads one arm's exact saved weights into `classify(queries)`.
Serve-safe arms (`no_branches` / `shuffled_targets` / `zipf_input`) classify raw strings;
`features_input` / `lightgbm` are saved but need the taxonomy extractor at serve time
(`serve_safe=False`), so raw-query classify is blocked. Serving policy stays gate + constant.

In [18]:
# 4c — load a saved arm; serve by tuned THRESHOLDS, then hedge close calls to pure_rrf
from sentence_transformers import SentenceTransformer

RRF_DELTA = 0.07   # override to pure_rrf when |p_dense - p_sparse| < RRF_DELTA (hedge on top of thresholds)

def serve_thr_rrf(probs, thr, delta=RRF_DELTA):
    """Tuned-threshold serving (unchanged), with a pure_rrf hedge overlaid on near-ties."""
    base = np.asarray(serve_from_probabilities(probs, thr))
    d = probs["dense_only"].to_numpy(); s = probs["sparse_only"].to_numpy()
    return np.where(np.abs(d - s) < delta, "pure_rrf", base)

def load_classifier(arm_dir):
    """Reload ONE arm's saved model into classify(queries) — no retraining."""
    p = _P(arm_dir); meta = _json.loads((p / "meta.json").read_text())
    svd = NgramSvd.load(p / "svd.joblib") if meta.get("svd_inputs", True) else None
    thr = np.load(p / "thresholds.npy")
    st = SentenceTransformer(meta["embedding_model"]); prefix = meta["prefix"]
    if meta["learner"] == "lgbm":
        models = joblib.load(p / "lgbm.joblib")
        prob = lambda X: pd.DataFrame(np.column_stack([m.predict_proba(X)[:, 1] for m in models]), columns=HEAD_ROUTES)
    elif meta["learner"] == "regressor":
        regs = joblib.load(p / "regressor.joblib")
        prob = lambda X: pd.DataFrame(np.column_stack([r.predict(X) for r in regs]), columns=HEAD_ROUTES)
    else:
        er = EncoderRouter.load(p / "router.pt"); prob = er.probabilities
    zstat = (np.load(p / "zipf_mean.npy"), np.load(p / "zipf_std.npy")) if meta["zipf_inputs"] else None
    sstat = (np.load(p / "shape_mean.npy"), np.load(p / "shape_std.npy")) if meta.get("shape_inputs") else None

    def classify(queries, delta=RRF_DELTA):
        if not meta["serve_safe"]:
            raise RuntimeError(f"{meta['arm']} uses taxonomy feature_inputs — assemble features first")
        q = [queries] if isinstance(queries, str) else list(queries)
        blocks = [np.asarray(st.encode([prefix + t for t in q], normalize_embeddings=True))]
        if svd is not None:
            blocks.append(svd.transform(pd.Series(q)))
        if zstat is not None:
            z = ZipfStats().frame(pd.Series(q)).to_numpy(np.float32)
            blocks.append((z - zstat[0]) / zstat[1])
        if sstat is not None:
            sh = LexicalShape().frame(pd.Series(q)).to_numpy(np.float32)
            blocks.append((sh - sstat[0]) / sstat[1])
        probs = prob(np.concatenate(blocks, axis=1).astype(np.float32))
        return probs.assign(route=serve_thr_rrf(probs, thr, delta), query=q)
    return classify

# classify = load_classifier(BASE / "zipf_input_nocorpus")   # reload any arm by name
# classify = load_classifier(BASE / "zipf_shape_nocorpus")

classify = load_classifier(BASE / "design_branches")

pd.set_option("display.max_colwidth", 70)
display(classify([
    "how do I refresh laravel migrations programmatically",
    "United States v. Fumo evidence Pennsylvania Ethics Act admissibility",
    "what is the difference between computer engineering and computer science",
]))

,sparse_only,dense_only,route,query
0,0.272017,0.528436,sparse_only,how do I refresh laravel migrations programmatically
1,0.546603,0.201457,sparse_only,United States v. Fumo evidence Pennsylvania Ethics Act admissibility
2,0.237497,0.732215,dense_only,what is the difference between computer engineering and computer s...


## 4d. Diagnostic battery — probe the dense/sparse decision boundary

Curated cases grouped by the mechanism they stress. `expect` is a mechanism-based HEURISTIC
(rare identifiers/codes → sparse; conceptual NL → dense; concept+rare-term → hybrid), NOT ground
truth — the interesting rows are the DIVERGENCES. The load-bearing category is `buried-id`: a rare
discriminating token inside fluent text. If the router serves dense there, it is reading fluency
and missing the rare token — the router-rarity-gap made visible on queries you control.

In [19]:
# 4d — diagnostic battery: does the router use lexical rarity, or default to dense on fluent text?
CASES = [
    ("id/code",   "sparse_only", "CVE-2021-44228 log4j remote code execution"),
    ("id/code",   "sparse_only", "ORA-00942 table or view does not exist"),
    ("id/code",   "sparse_only", "kubectl pod CrashLoopBackOff exit code 137"),
    ("id/code",   "sparse_only", "pip ImportError libGL.so.1 cannot open shared object file"),
    ("id/code",   "sparse_only", "doi:10.1038/s41586-020-2649-2"),
    ("buried-id", "sparse_only", "why does my postgres connection fail with ECONNREFUSED 127.0.0.1:5432"),
    ("buried-id", "sparse_only", "what causes the E11000 duplicate key error in mongodb"),
    ("buried-id", "sparse_only", "how do I fix Objects are not valid as a React child"),
    ("concept",   "dense_only",  "how does photosynthesis differ from cellular respiration"),
    ("concept",   "dense_only",  "what are the ethical implications of autonomous weapons"),
    ("concept",   "dense_only",  "explain the difference between empathy and sympathy"),
    ("concept",   "dense_only",  "why do people procrastinate even when they know the costs"),
    ("mixed",     "pure_rrf",    "difference between BM25 and TF-IDF for document ranking"),
    ("mixed",     "pure_rrf",    "how does the mRNA COVID-19 vaccine trigger an immune response"),
    ("mixed",     "pure_rrf",    "is Rust actually memory safe compared to C++"),
    ("exact",     "sparse_only", "\"I have a dream\" which speech and what year"),
    ("exact",     "sparse_only", "lyrics never gonna give you up never gonna let you down"),
    ("rare-1tok", "sparse_only", "defenestration"),
    ("oov",       "sparse_only", "was ist die Hauptstadt von Osterreich"),
    ("code",      "sparse_only", "for i in range(len(arr)): arr[i] += 1"),
    ("common",    "dense_only",  "what is the best way to be happy in life"),
    ("common",    "dense_only",  "how can I get better at my job over time"),
]
_cats = pd.DataFrame(CASES, columns=["category", "expect", "query"])
_res = classify(_cats["query"].tolist())
_pcols = [c for c in _res.columns if c in ("dense_only", "sparse_only", "pure_rrf")]
_res.insert(0, "category", _cats["category"].to_numpy())
_res.insert(1, "expect", _cats["expect"].to_numpy())
_res["match"] = _res["route"] == _res["expect"]

# CONSTANT baselines vs the SAME heuristic: a sparse-default scores well because the battery is sparse-heavy.
router_match = _res["match"].mean()
const_match = {r: float((_cats["expect"] == r).mean()) for r in ("sparse_only", "dense_only", "pure_rrf")}
best_const_route = max(const_match, key=const_match.get)
print(f"router match vs heuristic: {router_match:.0%}")
print(f"constant baselines: " + ", ".join(f"always-{r} {v:.0%}" for r, v in const_match.items()))
print(f"router LIFT over best constant (always-{best_const_route} {const_match[best_const_route]:.0%}): "
      f"{router_match - const_match[best_const_route]:+.0%}   <- this is the real signal, not the raw match")
print(f"router served mix: {_res['route'].value_counts(normalize=True).round(2).to_dict()} "
      f"(one route dominating => constant-in-disguise)")
pd.set_option("display.max_colwidth", 62)
display(_res[["category", "query", "expect", "route", "match"] + _pcols].round(3))
print("\nbalanced per-category match (buried-id low = rarity blindness):")
display(_res.groupby("category")["match"].agg(["mean", "count"]).round(2))

router match vs heuristic: 59%
constant baselines: always-sparse_only 59%, always-dense_only 27%, always-pure_rrf 14%
router LIFT over best constant (always-sparse_only 59%): +0%   <- this is the real signal, not the raw match
router served mix: {'sparse_only': 0.64, 'dense_only': 0.27, 'pure_rrf': 0.09} (one route dominating => constant-in-disguise)


,category,query,expect,route,match,sparse_only,dense_only
0,id/code,CVE-2021-44228 log4j remote code execution,sparse_only,sparse_only,True,0.309,0.440
1,id/code,ORA-00942 table or view does not exist,sparse_only,sparse_only,True,0.323,0.477
2,id/code,kubectl pod CrashLoopBackOff exit code 137,sparse_only,sparse_only,True,0.432,0.272
3,id/code,pip ImportError libGL.so.1 cannot open shared object file,sparse_only,sparse_only,True,0.380,0.525
4,id/code,doi:10.1038/s41586-020-2649-2,sparse_only,pure_rrf,False,0.458,0.422
5,buried-id,why does my postgres connection fail with ECONNREFUSED 127...,sparse_only,sparse_only,True,0.440,0.361
6,buried-id,what causes the E11000 duplicate key error in mongodb,sparse_only,sparse_only,True,0.426,0.353
7,buried-id,how do I fix Objects are not valid as a React child,sparse_only,dense_only,False,0.210,0.669
8,concept,how does photosynthesis differ from cellular respiration,dense_only,dense_only,True,0.219,0.634
9,concept,what are the ethical implications of autonomous weapons,dense_only,sparse_only,False,0.278,0.531



balanced per-category match (buried-id low = rarity blindness):


,mean,count
category,,
buried-id,0.67,3
code,0.00,1
common,1.00,2
concept,0.50,4
exact,1.00,2
id/code,0.80,5
mixed,0.00,3
oov,1.00,1
rare-1tok,0.00,1


In [20]:
display(classify([
    'Who likes Curling?',
    'what are the side effects of DHA',
    'CVE-2021-44228 log4j',
    'http://localhost.com',
    "why do cats purr",
    "Sheakspeare is great writer",
    "https://chatgpt.com/",
    "site:qdrant.tech documentation quickstart docker qdrant official",
    "I have a cat"
]))

,sparse_only,dense_only,route,query
0,0.999983,0.001767,sparse_only,Who likes Curling?
1,0.189865,0.777079,dense_only,what are the side effects of DHA
2,0.320072,0.398804,sparse_only,CVE-2021-44228 log4j
3,0.312757,0.502553,sparse_only,http://localhost.com
4,0.216074,0.682489,dense_only,why do cats purr
5,0.314839,0.576251,sparse_only,Sheakspeare is great writer
6,0.309971,0.469399,sparse_only,https://chatgpt.com/
7,0.451258,0.516575,pure_rrf,site:qdrant.tech documentation quickstart docker qdrant of...
8,0.217091,0.765552,dense_only,I have a cat


## 4e. Held-out evaluation — score ALL 5 saved arms (no refitting)

Loads every arm's saved artifact and scores it on `FAIR_LANE`'s DECISIVE rows. Feature/LightGBM arms
aren't raw-query serve-safe, but the held-out rows live in the frame, so their taxonomy/zipf inputs
are assembled from the saved z-stats — no extractor call, no refit. Leakage-free (asserts
`meta.holdout_lane == FAIR_LANE`); serving uses the diff-RRF rule (`RRF_DELTA`). Bar = best per-lane
constant; `served_max>=0.95` => constant-in-disguise; `shuffled_targets` is the permutation floor.

In [17]:
# 4e — score every SAVED arm on FAIR_LANE; compare to the per-lane ORACLE and the GLOBAL constant
RRF_DELTA = globals().get("RRF_DELTA", 0.15)
def _serve(probs, thr, delta=RRF_DELTA):        # tuned thresholds + pure_rrf hedge on near-ties
    base = np.asarray(serve_from_probabilities(probs, thr))
    d = probs["dense_only"].to_numpy(); sp = probs["sparse_only"].to_numpy()
    return np.where(np.abs(d - sp) < delta, "pure_rrf", base)

def global_constant(table, holdout_lane):
    """DEPLOYABLE baseline: single best route over ALL training lanes (no lane label at serve time)."""
    tr = table.frame[(table.frame["dataset"] != holdout_lane)
                     & (table.frame["shape"] == "routes_differ") & table.frame["serve"].notna()]
    means = {r: tr[f"score_{r}"].mean() for r in ROUTES}
    return max(means, key=means.get)

def evaluate_saved_frame(arm_dir, table, holdout_lane, global_route):
    p = _P(arm_dir); meta = _json.loads((p / "meta.json").read_text())
    assert meta.get("holdout_lane") == holdout_lane, (
        f"{p.name}: saved holdout={meta.get('holdout_lane')!r} != {holdout_lane!r} — retrain (leakage)")
    svd = NgramSvd.load(p / "svd.joblib") if meta.get("svd_inputs", True) else None
    thr = np.load(p / "thresholds.npy")
    if meta["learner"] == "lgbm":
        models = joblib.load(p / "lgbm.joblib")
        prob = lambda X: pd.DataFrame(np.column_stack([m.predict_proba(X)[:, 1] for m in models]), columns=HEAD_ROUTES)
    elif meta["learner"] == "regressor":
        regs = joblib.load(p / "regressor.joblib")
        prob = lambda X: pd.DataFrame(np.column_stack([r.predict(X) for r in regs]), columns=HEAD_ROUTES)
    else:
        er = EncoderRouter.load(p / "router.pt"); prob = er.probabilities

    frame = table.frame.reset_index(drop=True)
    emb = QueryEmbeddings(meta["embedding_model"]).matrix(frame)
    mask = ((frame["dataset"] == holdout_lane) & (frame["shape"] == "routes_differ")
            & frame["serve"].notna()).to_numpy()
    idx = np.flatnonzero(mask); q = frame.loc[idx, "query"]
    blocks = [emb[idx]] + ([svd.transform(q)] if svd is not None else [])
    if meta["feature_inputs"]:
        f = table.feature_matrix.to_numpy(np.float32)[idx]
        blocks.append((f - np.load(p / "feat_mean.npy")) / np.load(p / "feat_std.npy"))
    if meta["zipf_inputs"]:
        z = ZipfStats().frame(q).to_numpy(np.float32)
        blocks.append((z - np.load(p / "zipf_mean.npy")) / np.load(p / "zipf_std.npy"))
    if meta.get("shape_inputs"):
        sh = LexicalShape().frame(q).to_numpy(np.float32)
        blocks.append((sh - np.load(p / "shape_mean.npy")) / np.load(p / "shape_std.npy"))
    probs = prob(np.concatenate(blocks, axis=1).astype(np.float32))
    served = _serve(probs, thr)

    tf = frame.loc[idx]; sc = {r: tf[f"score_{r}"].to_numpy() for r in ROUTES}
    cap = np.array([sc[r][i] for i, r in enumerate(served)]).mean()
    consts = {r: sc[r].mean() for r in ROUTES}
    lane_best = max(consts, key=consts.get)          # per-lane ORACLE (needs lane label -> NOT deployable)
    lane_c = consts[lane_best]; glob_c = consts[global_route]   # GLOBAL constant (deployable everywhere)
    orc = np.column_stack([sc[r] for r in ROUTES]).max(1).mean()
    hr = lambda base: (cap - base) / (orc - base) if (orc - base) > 0.02 else np.nan
    return {"arm": p.name, "n_dec": len(idx), "captured": cap,
            "lane_route": lane_best, "lane_const": lane_c, "hr_vs_lane": hr(lane_c),
            "global_route": global_route, "global_const": glob_c,
            "lift_vs_global": cap - glob_c, "hr_vs_global": hr(glob_c),
            "oracle": orc, "served_max": float(pd.Series(served).value_counts(normalize=True).max())}

GLOBAL_ROUTE = global_constant(table_union, FAIR_LANE)
print(f"held-out {FAIR_LANE} | GLOBAL constant (deployable) = {GLOBAL_ROUTE!r} | "
      f"per-lane oracle route may differ")
rows = []
for a in PROBE_ARMS:
    d = BASE / a.name
    if not (d / "meta.json").exists():
        print(f"[{a.name}] not saved yet — run §3b"); continue
    try:
        rows.append(evaluate_saved_frame(d, table_union, FAIR_LANE, GLOBAL_ROUTE))
    except AssertionError as e:
        print("SKIP:", e)
print("\nhr_vs_global = the DEPLOYABLE bar (router beats the constant you could actually ship);")
print("hr_vs_lane   = the ORACLE bar (per-lane constant needs a lane label -> not shippable):")
display(pd.DataFrame(rows).round(3) if rows else "no saved models yet — run §3b")

held-out trec-dl-2022 | GLOBAL constant (deployable) = 'pure_rrf' | per-lane oracle route may differ

hr_vs_global = the DEPLOYABLE bar (router beats the constant you could actually ship);
hr_vs_lane   = the ORACLE bar (per-lane constant needs a lane label -> not shippable):


,arm,n_dec,captured,lane_route,lane_const,hr_vs_lane,global_route,global_const,lift_vs_global,hr_vs_global,oracle,served_max
0,no_branches,524,0.400,sparse_only,0.432,-0.119,pure_rrf,0.415,-0.015,-0.053,0.698,0.517
1,shuffled_targets,524,0.415,sparse_only,0.432,-0.063,pure_rrf,0.415,0.000,0.000,0.698,1.000
2,features_input_nocorpus,524,0.407,sparse_only,0.432,-0.096,pure_rrf,0.415,-0.009,-0.031,0.698,0.492
3,zipf_input_nocorpus,524,0.400,sparse_only,0.432,-0.120,pure_rrf,0.415,-0.015,-0.053,0.698,0.607
4,zipf_shape_nocorpus,524,0.373,sparse_only,0.432,-0.224,pure_rrf,0.415,-0.043,-0.152,0.698,0.660
5,lightgbm_nocorpus,524,0.417,sparse_only,0.432,-0.057,pure_rrf,0.415,0.001,0.005,0.698,0.429
6,design_branches,524,0.429,sparse_only,0.432,-0.013,pure_rrf,0.415,0.013,0.046,0.698,0.490
7,no_svd_nocorpus,524,0.398,sparse_only,0.432,-0.127,pure_rrf,0.415,-0.017,-0.060,0.698,0.552
8,score_regressor_nocorpus,524,0.388,sparse_only,0.432,-0.165,pure_rrf,0.415,-0.027,-0.096,0.698,0.740
9,decisive_only,524,0.391,sparse_only,0.432,-0.155,pure_rrf,0.415,-0.025,-0.087,0.698,0.469


In [32]:
# 4h — COST_STEP=0 ablation: re-tune thresholds cost-free on the SAME saved models.
#       Cost only affects thresholds, not weights -> NO retrain. The regressor is already
#       cost-free (thr=[0,0]); it is the reference row. Read: if cost=0 pushes served_max->1.0
#       and hr_vs_global->~0, the cost term was propping up route diversity, not the model.
from encoder_router.evaluate import COST_STEP as _DEFAULT_COST, tuned_thresholds

def _assemble(meta, p, frame, idx):
    """Rebuild an arm's exact input matrix for the given rows from its saved transforms."""
    blocks = [QueryEmbeddings(meta["embedding_model"]).matrix(frame)[idx]]
    q = frame.loc[idx, "query"]
    if meta.get("svd_inputs", True):
        blocks.append(NgramSvd.load(p / "svd.joblib").transform(q))
    if meta["feature_inputs"]:
        f = table_union.feature_matrix.to_numpy(np.float32)[idx]
        blocks.append((f - np.load(p / "feat_mean.npy")) / np.load(p / "feat_std.npy"))
    if meta["zipf_inputs"]:
        z = ZipfStats().frame(q).to_numpy(np.float32)
        blocks.append((z - np.load(p / "zipf_mean.npy")) / np.load(p / "zipf_std.npy"))
    if meta.get("shape_inputs"):
        sh = LexicalShape().frame(q).to_numpy(np.float32)
        blocks.append((sh - np.load(p / "shape_mean.npy")) / np.load(p / "shape_std.npy"))
    return np.concatenate(blocks, axis=1).astype(np.float32)

def _prob_fn(meta, p):
    if meta["learner"] == "lgbm":
        models = joblib.load(p / "lgbm.joblib")
        return lambda X: pd.DataFrame(np.column_stack([m.predict_proba(X)[:, 1] for m in models]), columns=HEAD_ROUTES)
    if meta["learner"] == "regressor":
        regs = joblib.load(p / "regressor.joblib")
        return lambda X: pd.DataFrame(np.column_stack([r.predict(X) for r in regs]), columns=HEAD_ROUTES)
    return EncoderRouter.load(p / "router.pt").probabilities

def eval_arm_costs(arm_dir, cost_steps, holdout=FAIR_LANE, global_route=None):
    """Score one saved arm at each cost_step (thresholds re-tuned on the training pool)."""
    global_route = global_route or GLOBAL_ROUTE
    p = _P(arm_dir); meta = _json.loads((p / "meta.json").read_text())
    frame = table_union.frame.reset_index(drop=True); prob = _prob_fn(meta, p)
    differ = (frame["shape"] == "routes_differ") & frame["serve"].notna()
    idx = np.flatnonzero((frame["dataset"] == holdout).to_numpy() & differ.to_numpy())
    probs_test = prob(_assemble(meta, p, frame, idx))
    sc = {r: frame.loc[idx, f"score_{r}"].to_numpy() for r in ROUTES}
    orc = np.column_stack([sc[r] for r in ROUTES]).max(1).mean(); gc = sc[global_route].mean()
    def row(cs, thr):
        served = _serve(probs_test, thr); cap = np.array([sc[r][i] for i, r in enumerate(served)]).mean()
        return {"arm": p.name, "cost_step": round(float(cs), 3), "thr": np.round(thr, 2).tolist(),
                "captured": cap, "global_const": gc,
                "hr_vs_global": (cap - gc) / (orc - gc) if (orc - gc) > 0.02 else np.nan,
                "served_max": float(pd.Series(served).value_counts(normalize=True).max())}
    if meta["learner"] == "regressor":
        return [row(0.0, np.zeros(len(HEAD_ROUTES), np.float32))]     # cost-invariant by construction
    pool = np.flatnonzero((frame["dataset"] != holdout).to_numpy() & differ.to_numpy())
    probs_pool = prob(_assemble(meta, p, frame, pool)).reset_index(drop=True)
    fpool = frame.iloc[pool].reset_index(drop=True)
    return [row(cs, tuned_thresholds(probs_pool, fpool, cost_step=cs)) for cs in cost_steps]

_rows = []
for a in PROBE_ARMS:
    d = BASE / a.name
    if (d / "meta.json").exists():
        _rows.extend(eval_arm_costs(d, (_DEFAULT_COST, 0.0)))
print(f"COST_STEP ablation | held-out {FAIR_LANE!r} | default cost={_DEFAULT_COST:.3f} vs 0.0")
print("default-cost rows should match §4e; cost=0 rows are the cost-free quality-only tuning")
display(pd.DataFrame(_rows).round(3))

COST_STEP ablation | held-out 'trec-dl-2022' | default cost=0.300 vs 0.0
default-cost rows should match §4e; cost=0 rows are the cost-free quality-only tuning


,arm,cost_step,thr,captured,global_const,hr_vs_global,served_max
0,no_branches,0.3,"[0.35, 0.9]",0.420,0.415,0.015,0.574
1,no_branches,0.0,"[0.4, 0.5]",0.356,0.415,-0.211,0.840
2,shuffled_targets,0.3,"[0.3, 0.55]",0.415,0.415,0.000,1.000
3,shuffled_targets,0.0,"[0.3, 0.55]",0.415,0.415,0.000,1.000
4,features_input_nocorpus,0.3,"[0.3, 0.9]",0.412,0.415,-0.011,0.710
5,features_input_nocorpus,0.0,"[0.45, 0.6]",0.373,0.415,-0.152,0.845
6,zipf_input_nocorpus,0.3,"[0.3, 0.85]",0.416,0.415,0.001,0.668
7,zipf_input_nocorpus,0.0,"[0.45, 0.5]",0.351,0.415,-0.226,0.847
8,zipf_shape_nocorpus,0.3,"[0.3, 0.9]",0.411,0.415,-0.014,0.634
9,zipf_shape_nocorpus,0.0,"[0.45, 0.55]",0.369,0.415,-0.163,0.828


## 4f. Zipf-rule router — the simplest thing (serve-safe, no model)

Rare-token rule: if the query's **rarest** token falls below the conventional rare line (`ZIPF_RARE` = 3.0, ~once per million words) route **sparse** (lexical / identifiers), else **dense**. Uses `zipf.min`, not a rare-*share*, so a single OOV identifier in fluent prose ("…error `SQLSTATE 23505`") still triggers lexical instead of being averaged away by the common words around it. Behavioral acceptance is scored in §4g; the retrieval bar is below.

In [29]:
# 4f - Zipf-rule router (serve-safe, no model): route sparse when the query's
# RAREST token is below the conventional rare line, else dense. zipf.min (not a
# rare-SHARE) so one identifier in fluent prose isn't diluted away.
from encoder_router.table import ZipfStats, ZIPF_RARE

Z_RARE_MIN = ZIPF_RARE   # 3.0 = once/million words; a token below it is rare/OOV

def load_classifier_with_zipf(rare_min=Z_RARE_MIN):
    zs = ZipfStats()
    def classify(queries):
        q = [queries] if isinstance(queries, str) else list(queries)
        zf = zs.frame(pd.Series(q))
        route = np.where(zf["zipf.min"].to_numpy() < rare_min, "sparse_only", "dense_only")
        return pd.DataFrame({"query": q, "route": route}).join(zf.round(2))
    return classify

def evaluate_zipf_rule(table, holdout_lane, **kw):
    clf = load_classifier_with_zipf(**kw)
    tf = table.frame[(table.frame["dataset"] == holdout_lane)
                     & (table.frame["shape"] == "routes_differ") & table.frame["serve"].notna()]
    served = clf(tf["query"].tolist())["route"].to_numpy()
    sc = {r: tf[f"score_{r}"].to_numpy() for r in ROUTES}
    cap = np.array([sc[r][i] for i, r in enumerate(served)]).mean()
    consts = {r: sc[r].mean() for r in ROUTES}
    best = max(consts, key=consts.get); orc = np.column_stack([sc[r] for r in ROUTES]).max(1).mean()
    return {"rule": "zipf_min", "n_decisive": len(tf), "best_route": best,
            "headroom": (cap - consts[best]) / (orc - consts[best]) if (orc - consts[best]) > 0.02 else np.nan,
            "raw_lift": cap - consts[best], "captured": cap, "best_const": consts[best], "oracle": orc,
            "served_max": float(pd.Series(served).value_counts(normalize=True).max())}

zipf_router = load_classifier_with_zipf()
pd.set_option("display.max_colwidth", 55)
print("demo (behavioral rule: rarest token < 3.0 -> sparse):")
display(zipf_router([
    "CVE-2021-44228 log4j remote code execution", "http://localhost.com", "Who likes Curling?",
    "what are the side effects of DHA", "how does photosynthesis differ from cellular respiration",
]))
print(f"\nbehavioral acceptance -> see 4g (cases.json). Retrieval bar on {FAIR_LANE} "
      f"(beat best_const, served_max<0.95):")
display(pd.DataFrame([evaluate_zipf_rule(table_union, FAIR_LANE)]).round(3))


demo (behavioral rule: rarest token < 3.0 -> sparse):


,query,route,zipf.min,zipf.mean,zipf.max,zipf.rare_share,zipf.oov_share
0,CVE-2021-44228 log4j remote code execution,sparse_only,0.00,3.14,5.08,0.43,0.14
1,http://localhost.com,sparse_only,2.02,3.68,4.81,0.33,0.00
2,Who likes Curling?,dense_only,3.51,4.86,6.34,0.00,0.00
3,what are the side effects of DHA,sparse_only,2.78,5.92,7.73,0.14,0.00
4,how does photosynthesis differ from cellular respir...,dense_only,3.03,4.66,6.63,0.00,0.00



behavioral acceptance -> see 4g (cases.json). Retrieval bar on trec-dl-2022 (beat best_const, served_max<0.95):


,rule,n_decisive,best_route,headroom,raw_lift,captured,best_const,oracle,served_max
0,zipf_min,524,sparse_only,-0.111,-0.03,0.403,0.432,0.698,0.634


## 4g. Behavioral test of the §4f Zipf-rule on `cases.json`

The frozen fixtures are the real bar for a *behavioral* rule: model-free, no scores — does each route land in that query's acceptable set? Arms for reference (from the audit): base **86/108**, zipf-MLP **80/108**; user originals base 2/5, zipf-MLP 5/5. Watch `identifier_in_prose` — a share-based trigger dilutes a lone identifier across fluent prose and misroutes it to dense.

In [30]:
# 4g - behavioral acceptance of the 4f Zipf-rule on the frozen cases.json fixtures.
# Model-free, no scores: does each served route land in that query's acceptable set?
import json
from IPython.display import display

SHORT = {"D": "dense_only", "S": "sparse_only", "R": "pure_rrf"}
CASES = next(p for p in (
    DATA_DIR.parents[1] / "experiments" / "router_behavior_audit" / "cases.json",
    Path.cwd() / "experiments" / "router_behavior_audit" / "cases.json",
) if p.exists())
cases = json.loads(CASES.read_text())

fresh = pd.DataFrame([
    dict(category=g["category"], family=g["family"], variant=v, query=q,
         acceptable=[SHORT[a] for a in g["acceptable"]])
    for g in cases["groups"] for v, q in enumerate(g["queries"])
])
fresh["route"] = zipf_router(fresh["query"].tolist())["route"].to_numpy()  # the 4f rule
fresh["accepted"] = [r in ok for r, ok in zip(fresh["route"], fresh["acceptable"])]

by_cat = fresh.groupby("category").agg(
    n=("accepted", "size"), accepted=("accepted", "mean"),
    routes=("route", lambda s: s.value_counts().to_dict()))
stability = fresh.groupby("family")["route"].nunique().eq(1).mean()

users = pd.DataFrame([dict(query=r["query"], expected=SHORT[r["expected"]])
                      for r in cases["user_examples"]])
users["route"] = zipf_router(users["query"].tolist())["route"].to_numpy()
users["ok"] = users["route"].eq(users["expected"])

print(f"fresh acceptance: {fresh['accepted'].mean():.3f} "
      f"({int(fresh['accepted'].sum())}/{len(fresh)})    ref: base 86/108, zipf-MLP 80/108")
print(f"paraphrase families with ONE route across 3 phrasings: {stability:.3f}")
print(f"user originals: {int(users['ok'].sum())}/5    ref: base 2/5, zipf-MLP 5/5\n")
display(by_cat.round(3)); display(users)


fresh acceptance: 0.944 (102/108)    ref: base 86/108, zipf-MLP 80/108
paraphrase families with ONE route across 3 phrasings: 1.000
user originals: 2/5    ref: base 2/5, zipf-MLP 5/5



,n,accepted,routes
category,,,
concept,36,1.000,{'dense_only': 36}
exact_lookup,36,0.917,"{'sparse_only': 33, 'dense_only': 3}"
identifier_in_prose,36,0.917,"{'sparse_only': 33, 'dense_only': 3}"


,query,expected,route,ok
0,Who likes Curling?,sparse_only,dense_only,False
1,what are the side effects of DHA,sparse_only,sparse_only,True
2,CVE-2021-44228 log4j,pure_rrf,sparse_only,False
3,http://localhost.com,sparse_only,sparse_only,True
4,why do cats purr,dense_only,sparse_only,False
